# The price as an input: do the models know anything the price doesn't?

Companion notebook to the [market-as-input report](https://github.com/JHvL88/matchday/blob/main/analysis/backtest/blend/README.md) of [Jan's Matchday](https://jhvl88.github.io/matchday/).
Earlier tests asked whether the models beat the bookmakers. This one asks less: give a forecaster the bookmakers' price, then show it the models as well. Does it forecast better?
The questions and rules were [fixed beforehand](https://github.com/JHvL88/matchday/blob/main/analysis/backtest/blend/PREREGISTRATION.md).

1. **Source data** for five leagues and one match table per league.
2. **Forecasts** as published: the three models on all five leagues, and the Bundesliga's xG models.
3. **The blend** of price and models: its weights, how it forecasts matches it hasn't seen, and what betting on it would have returned.

**Runtime:** about ten minutes on a CPU with `RESAMPLES = 4000`, as in the report; nearly all of it is refitting the blend 4,000 times for each interval. `RESAMPLES = 400` gives the same picture in about two minutes. No model is refitted, so no GPU is needed; [`leagues.ipynb`](https://github.com/JHvL88/matchday/blob/main/analysis/backtest/leagues/leagues.ipynb) refits the models themselves.

Set `LATEST = True` to grade today's source files and forecasts; the numbers then drift from the report as matches are added.

In [ ]:
LATEST = False         # True: the newest source files and forecasts, not the ones the report graded
RESAMPLES = 4000       # bootstrap refits behind each weight's interval; 400 is five times faster

import json, os, subprocess, sys
REPO = "https://github.com/JHvL88/matchday"
BRANCH = os.environ.get("MATCHDAY_BRANCH", "main")
LEAGUES = ["GER", "ENG", "ESP", "ITA", "FRA"]
SOURCE_COMMIT = "08f400c56187a033afd7bc4d50a8e78a780bd154"     # source-data commit behind the report
RESULTS_COMMIT = "f18b825b8a79ffc03435d82f0f3af4e655cad9eb"    # backtest-results commit behind the report

def sh(*cmd, quiet=False):
    r = subprocess.run([str(c) for c in cmd], capture_output=True, text=True)
    if (not quiet or r.returncode) and (r.stdout + r.stderr).strip():
        print((r.stdout + r.stderr).strip())
    if r.returncode:
        raise RuntimeError(f"failed: {' '.join(str(c) for c in cmd)}")
    return r.stdout

def fetch(folder, commit, branch):
    """One commit of the repository (or the tip of a branch) into a folder."""
    if not os.path.exists(folder):
        sh("git", "init", "-q", folder)
        sh("git", "-C", folder, "fetch", "-q", "--depth", "1", REPO, branch if LATEST else commit)
        sh("git", "-C", folder, "checkout", "-q", "FETCH_HEAD")

if not os.path.exists("matchday"):
    sh("git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO, "matchday")
fetch("source", SOURCE_COMMIT, "source-data")
fetch("published", RESULTS_COMMIT, "backtest-results")
PIPE = "matchday/analysis/pipeline"
sys.path.insert(0, PIPE)
print("code:", sh("git", "-C", "matchday", "log", "-1", "--format=%h %s", quiet=True).strip())
print("source data:", sh("git", "-C", "source", "log", "-1", "--format=%h %s", quiet=True).strip())
print("forecasts:", sh("git", "-C", "published", "log", "-1", "--format=%h %s", quiet=True).strip())

### One match table per league

Each row keeps the early price (`avg_*`, collected a few days before the match), the closing price (`avgc_*`) and a pointer to its source file and line, which is how the best early price is looked up later.

In [ ]:
import pandas as pd
os.makedirs("tables", exist_ok=True)
rows = []
for league in LEAGUES:
    sh(sys.executable, f"{PIPE}/build_matches.py", "source", f"tables/{league}.csv", "--league", league, quiet=True)
    m = pd.read_csv(f"tables/{league}.csv", parse_dates=["date"])
    rows.append(dict(league=league, matches=len(m), with_early_price=int(m.avg_h.notna().sum()),
                     with_closing_price=int(m.avgc_h.notna().sum()), last_match=f"{m.date.max():%Y-%m-%d}"))
pd.DataFrame(rows)

### The blend, in a dozen lines

Each outcome of a match (home, draw, away) gets a score, and the three scores become probabilities that sum to one:

    score = a + b × ln(market probability) + c × ln(model probability)

`a` is one constant for draws and one for away wins. With `a` = 0, `b` = 1, `c` = 0 the blend is the market price unchanged.

- **`b`** above 1: favourites win more often than the price says.
- **`c`** is the question. At 0 the models add nothing to the price. Above 0 the best forecast leans toward the models, below 0 away from them.

The cell fits the blend on every match with nothing but numpy and scipy, then repeats the exercise honestly: each season from 2020/21 is forecast by a blend fitted on earlier seasons only. **Log loss** is minus the logarithm of the probability a forecast gave to what happened; lower is better.

In [ ]:
import numpy as np
from scipy.optimize import minimize
from evaluate import cols, load

o = pd.concat([load(f"tables/{league}.csv", f"published/leagues/{league}") for league in LEAGUES])
o = o.dropna(subset=cols("bay") + cols("avgc") + cols("avg")).reset_index(drop=True)
market, model = np.log(o[cols("avg")].to_numpy()), np.log(o[cols("ens")].to_numpy())
y, season = o.y.to_numpy(), o.season.to_numpy()

def blend(t, rows):
    """Probabilities from the weights t = (a_draw, a_away, b, c)."""
    score = t[2] * market[rows] + t[3] * model[rows] + np.array([0, t[0], t[1]])
    e = np.exp(score - score.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)

def log_loss(p, rows):
    return -np.log(p[np.arange(len(p)), y[rows]])

def fit(rows, with_models=True):
    free = [0, 1, 2, 3] if with_models else [0, 1, 2]
    def loss(x):
        t = np.array([0, 0, 1, 0.0]); t[free] = x
        return log_loss(blend(t, rows), rows).sum()
    t = np.array([0, 0, 1, 0.0]); t[free] = minimize(loss, t[free], method="BFGS").x
    return t

everything = np.ones(len(o), bool)
t0, t1 = fit(everything, with_models=False), fit(everything)
print(f"{len(o):,} matches")
print(f"price only:        b = {t0[2]:.2f}, draws {t0[0]:+.2f}, away wins {t0[1]:+.2f}")
print(f"price and models:  b = {t1[2]:.2f}, c = {t1[3]:+.2f}   <- the weight on the models")

gain = []
for s in sorted(set(season))[1:]:
    before, now = season < s, season == s
    gain.append(log_loss(blend(fit(before), now), now) - log_loss(blend(fit(before, with_models=False), now), now))
gain = np.concatenate(gain)
print(f"{len(gain):,} matches the blend hadn't seen: log loss with the models minus without = {gain.mean():+.5f}")

### Grades

`evaluate_blend.py` does the same with an interval around every number (a bootstrap over matches), for each model, league and season, for the closing price, and for the Bundesliga's published models. It then applies the pre-registered rules:

- **Q1:** *Information* if `c` is above zero with its whole interval and the blend forecasts unseen matches better with the models; *Contrary* if `c` is below zero and it still forecasts better; otherwise *Nothing*.
- **Q2:** €1 on every outcome where blend probability × odds is at least 1. *Value* if the realised return at the average early price is above zero with its whole interval; *value only by shopping around* if that holds at the best price in the market; otherwise *No value*.

In [ ]:
print(sh(sys.executable, f"{PIPE}/evaluate_blend.py", "tables", "source", "published/leagues", "results.json",
         "--xg-results", "published/xg", "--resamples", RESAMPLES, quiet=True))
res = json.load(open("results.json"))
pub = json.load(open("matchday/analysis/backtest/blend/results.json"))
pick = lambda r: (r["q1"]["weight"]["value"], r["q1"]["gain"]["value"], r["q2"]["average_price"]["realised_return"]["mean"],
                  r["q2"]["average_price"]["bets"])
same = all(abs(a - b) < 1e-9 for a, b in zip(pick(res), pick(pub)))
print("Same answers as the published report." if same else
      "Differs from the published report (expected with LATEST = True):", "" if same else (pick(res), pick(pub)))

### The weight on the models

One row per forecast tested. Every weight is zero or below; none helps on matches the blend hadn't seen.

In [ ]:
A = res["five_leagues"]
iv = lambda v, d=2: f"{v['value']:+.{d}f} ({v['low']:+.{d}f} to {v['high']:+.{d}f})"
def weight_row(label, block):
    w = block["weights"]["price_and_models"]
    gain = block["unseen"]["gain"] if "unseen" in block else block["gain"]
    return {"forecast": label, "matches": block["weights"]["matches"], "b (market)": iv(w["market"]), "c (models)": iv(w["models"]),
            "unseen gain, log loss": iv(gain["log_loss"], 5)}
pd.DataFrame([weight_row("The ensemble, early price", A),
              weight_row("Dixon-Coles alone", A["per_model"]["dixon_coles"]),
              weight_row("Bayesian alone", A["per_model"]["bayesian"]),
              weight_row("XGBoost alone", A["per_model"]["xgboost"]),
              weight_row("Ensemble with XGBoost trained on five leagues", A["per_model"]["ensemble_pooled_xgboost"]),
              weight_row("The ensemble next to the closing price", A["closing_price"]),
              weight_row("Bundesliga, published models", res["bundesliga_published"])])

### The betting rule

Realised return decides. "Price only" is the same rule on the blend without the models: bets that owe nothing to them, only to the price's own habits. "Every bet" is all three outcomes of every match.

In [ ]:
u = A["unseen"]
def bet_row(label, r):
    f = lambda k, d=1: f"{r[k]['mean'] * 100:+.{d}f}% ({r[k]['low'] * 100:+.{d}f} to {r[k]['high'] * 100:+.{d}f})"
    return {"bets": label, "number": r["bets"], "average odds": round(r["odds"], 2), "realised return": f("realised_return"),
            "CLV": f("clv", 2), "home / draw / away": " / ".join(str(v) for v in r["outcomes"].values())}
pd.DataFrame([bet_row("Price and models, average early price", u["rule"]["avg"]),
              bet_row("Price and models, Bet365's early price", u["rule"]["b365"]),
              bet_row("Price and models, best early price", u["rule"]["max"]),
              bet_row("Price only, average early price", u["price_only_rule"]["avg"]),
              bet_row("Price only, best early price", u["price_only_rule"]["max"]),
              bet_row("Every possible bet, average early price", u["every_bet"]["avg"]),
              bet_row("Every possible bet, best early price", u["every_bet"]["max"])])

In [ ]:
from IPython.display import SVG, display
sh(sys.executable, f"{PIPE}/make_blend_figures.py", "results.json", "figs", quiet=True)
for name in ["weights", "gain", "returns"]:
    display(SVG(f"figs/{name}.svg"))
print(f"Q1: {res['q1']['verdict'].upper()}.  Q2: {res['q2']['verdict'].upper()}.")